# Pandas Application & Feature Extraction Assignment
**File Name:** `panda_assignment.ipynb`  
**Dataset:** `anime.csv`


### 1. Load Data
Reading `anime.csv` dataset.


In [1]:
import numpy as np
import pandas as pd
import re

# Load anime dataset
anime_raw_df = pd.read_csv('anime.csv')
anime_raw_df.head()


,Rank,Title,Score
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05


### 2. Dataset Overview
Checking dataset shape and summary info.


In [2]:
# Inspect shape and column info
data_dimensions = anime_raw_df.shape
print(f"Rows: {data_dimensions[0]}, Columns: {data_dimensions[1]}\n")
anime_raw_df.info()


Rows: 50, Columns: 3

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Rank    50 non-null     int64  
 1   Title   50 non-null     object 
 2   Score   50 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.3+ KB


### 3. Feature Extraction - Episodes Count
Extracting episode numbers from the `Title` column.


In [3]:
# Extract episodes count using string parsing
def get_episodes(text):
    match = re.search(r'\((\d+)\s*eps\)', str(text))
    if match:
        return int(match.group(1))
    return 1

parsed_anime_df = anime_raw_df.copy()
parsed_anime_df['Episodes'] = parsed_anime_df['Title'].apply(get_episodes)
parsed_anime_df[['Rank', 'Score', 'Episodes']].head()


,Rank,Score,Episodes
0,1,9.10,64
1,2,9.07,24
2,3,9.06,13
3,4,9.06,51
4,5,9.05,10


### 4. Feature Extraction - Anime Type
Extracting show type (TV, Movie, OVA, Special) from `Title`.


In [4]:
# Extract anime format type
def get_type(text):
    for kind in ['TV', 'Movie', 'OVA', 'Special']:
        if kind in str(text):
            return kind
    return 'TV'

parsed_anime_df['Type'] = parsed_anime_df['Title'].apply(get_type)
parsed_anime_df['Type'].value_counts()


Type
TV         38
Movie       9
OVA         2
Special     1
Name: count, dtype: int64

### 5. Feature Extraction - Members Count
Extracting member count from `Title`.


In [5]:
# Extract member count
def get_members(text):
    match = re.search(r'([\d,]+)\s*members', str(text))
    if match:
        return int(match.group(1).replace(',', ''))
    return np.nan

parsed_anime_df['Members'] = parsed_anime_df['Title'].apply(get_members)
parsed_anime_df[['Title', 'Members']].head()


,Title,Members
0,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,20103218472
1,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",20112473707
2,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,2022474138
3,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",2016605113
4,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,20192146679


### 6. Cleaning Title Column
Extracting clean title text by removing format, episodes, dates, and member tags.


In [6]:
# Extract clean anime title name
def get_clean_title(text):
    # Remove format tag and everything after
    parts = re.split(r'(TV|Movie|OVA|Special)', str(text))
    clean = parts[0].strip().strip('"')
    return clean

cleaned_anime_df = parsed_anime_df.copy()
cleaned_anime_df['Clean_Title'] = cleaned_anime_df['Title'].apply(get_clean_title)
cleaned_anime_df[['Rank', 'Clean_Title', 'Type', 'Episodes', 'Score', 'Members']].head(10)


,Rank,Clean_Title,Type,Episodes,Score,Members
0,1,Fullmetal Alchemist: Brotherhood,TV,64,9.10,20103218472
1,2,Steins;Gate,TV,24,9.07,20112473707
2,3,Bleach: Sennen Kessen-hen,TV,13,9.06,2022474138
3,4,Gintama°,TV,51,9.06,2016605113
4,5,Shingeki no Kyojin Season 3 Part 2,TV,10,9.05,20192146679
5,6,Gintama',TV,51,9.04,2012534105
6,7,Gintama: The Final,Movie,1,9.04,2021137208
7,8,Hunter x Hunter,TV,148,9.04,20142701154
8,9,Kaguya-sama wa Kokurasetai: Ultra Romantic,TV,13,9.04,2022851445
9,10,Gintama': Enchousen,TV,13,9.03,2013313446


### 7. Highest Scoring Anime Analysis
Finding anime with the top scores.


In [7]:
# Top 5 highest scoring anime
top_scored_anime = cleaned_anime_df.sort_values(by='Score', ascending=False)[['Rank', 'Clean_Title', 'Score', 'Type']].head(5)
top_scored_anime


,Rank,Clean_Title,Score,Type
0,1,Fullmetal Alchemist: Brotherhood,9.10,TV
1,2,Steins;Gate,9.07,TV
2,3,Bleach: Sennen Kessen-hen,9.06,TV
3,4,Gintama°,9.06,TV
4,5,Shingeki no Kyojin Season 3 Part 2,9.05,TV


### 8. Longest Running Anime (Highest Episode Count)
Finding anime with the most episodes.


In [8]:
# Top 5 anime with most episodes
longest_anime = cleaned_anime_df.sort_values(by='Episodes', ascending=False)[['Clean_Title', 'Episodes', 'Type', 'Score']].head(5)
longest_anime


,Clean_Title,Episodes,Type,Score
15,Gintama,201,TV,8.94
7,Hunter x Hunter,148,TV,9.04
11,Ginga Eiyuu Densetsu,110,OVA,9.02
42,Hajime no Ippo,75,TV,8.76
24,Monster,74,TV,8.87


### 9. Most Popular Anime (Highest Member Count)
Finding anime with highest member engagement.


In [9]:
# Top 5 most popular anime
popular_anime_df = cleaned_anime_df.sort_values(by='Members', ascending=False)[['Clean_Title', 'Members', 'Score', 'Type']].head(5)
popular_anime_df


,Clean_Title,Members,Score,Type
35,Kimetsu no Yaiba: Yuukaku-hen,20221302152,8.79,TV
41,Shingeki no Kyojin: The Final Season Part 2,20221170165,8.77,TV
34,Shingeki no Kyojin: The Final Season,20211882760,8.80,TV
4,Shingeki no Kyojin Season 3 Part 2,20192146679,9.05,TV
33,Mob Psycho 100 II,20191458089,8.80,TV


### 10. Summary Statistics by Type
Grouping scores and episodes by anime format type.


In [10]:
# Aggregated statistics by Type
type_summary_df = cleaned_anime_df.groupby('Type').agg({
    'Score': ['mean', 'max', 'min'],
    'Episodes': ['mean', 'max'],
    'Clean_Title': 'count'
}).reset_index()

type_summary_df


Type     Score               Episodes      Clean_Title
                mean   max   min       mean  max       count
0    Movie  8.873333  9.04  8.78   1.000000    1           9
1      OVA  8.865000  9.02  8.71  57.000000  110           2
2  Special  9.030000  9.03  9.03   2.000000    2           1
3       TV  8.882632  9.10  8.71  30.868421  201          38

### Completed
Pandas feature extraction and data manipulation assignment completed.
